# Drug Safety Signal Summary

Reads `pharmacovigilance.adverse_events` (see
`data-generators/fda_adverse_event_ingest.ipynb`) and aggregates by
drug/reaction pair — a pharmacovigilance team's core question: which
drug-reaction combinations are showing up most often, and how serious are
they, so rising signals get flagged before they become a regulatory
issue.

Re-runnable: `mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "pharmacovigilance"
SOURCE_TABLE = "adverse_events"
TARGET_TABLE = "drug_safety_signal_summary_v1"

## Load bronze table

In [ ]:
df = spark.table(f"{DATABASE_NAME}.{SOURCE_TABLE}")

## Transform: safety signal by drug/reaction

Aggregates by drug name and reaction term: report count, serious-report
count, death count, and serious rate. `serious`/`seriousness_death` come
through the API as `"1"`/`"2"`-style strings — `"1"` means yes.

In [ ]:
summary_df = (
    df
    .groupBy("drug_name", "reaction_term")
    .agg(
        F.count("*").alias("report_count"),
        F.sum(F.when(F.col("serious") == "1", 1).otherwise(0)).alias("serious_count"),
        F.sum(F.when(F.col("seriousness_death") == "1", 1).otherwise(0)).alias("death_count"),
    )
    .withColumn("serious_rate_pct", F.round(F.col("serious_count") * 100.0 / F.col("report_count"), 1))
    .orderBy(F.desc("report_count"))
)

## Persist as a managed Spark table

In [ ]:
summary_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TARGET_TABLE}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_signals FROM {DATABASE_NAME}.{TARGET_TABLE}").show()
spark.sql(f"SELECT * FROM {DATABASE_NAME}.{TARGET_TABLE} ORDER BY report_count DESC LIMIT 10").show()